# Tune `mspc_lr`

MSPC + elastic-net logistic. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/mspc_lr.json`](../data/processed/tuned/mspc_lr.json).

**Stage 1 (hyperparameters):** PLS `n_components`, classifier `C`, `l1_ratio`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "mspc_lr"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__pls__n_components,classifier__C,classifier__l1_ratio
0,"[15, 20, 25, 30]",NaN,NaN
1,NaN,"[0.1, 1.0]",NaN
2,NaN,NaN,"[0.5, 0.95]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


mspc_lr: 16 candidates x 25 folds = 400 fits


GridSearchCV 400 fits:   0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Fitting 25 folds for each of 16 candidates, totalling 400 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,n_components,c,l1_ratio,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
5,20,0.1,0.95,37.039530,5.665531,0.629605,41.750000,11.912709,84.170940,2.442950,0.706026,0.059444,0.192287,0.052162
9,25,0.1,0.95,36.894231,6.178876,0.631058,41.750000,13.104781,84.461538,2.432161,0.702697,0.059510,0.191834,0.055853
1,15,0.1,0.95,36.889140,5.550973,0.631109,42.529412,12.097642,83.692308,2.920227,0.705748,0.060052,0.191154,0.055010
13,30,0.1,0.95,36.346280,6.310494,0.636537,42.264706,13.406765,85.042735,2.261326,0.700956,0.056957,0.191075,0.055257
0,15,0.1,0.50,36.326546,5.712663,0.636735,43.705882,12.350893,83.641026,2.571272,0.704285,0.058766,0.189805,0.056811
12,30,0.1,0.50,35.957956,6.029790,0.640420,42.955882,12.855532,85.128205,2.192437,0.700927,0.054578,0.188257,0.052285
4,20,0.1,0.50,36.257919,5.457816,0.637421,43.176471,11.567782,84.307692,2.323529,0.706983,0.059921,0.187630,0.051737
8,25,0.1,0.50,36.375943,5.761818,0.636241,42.735294,11.791000,84.512821,2.291109,0.702561,0.057690,0.186825,0.054609
3,15,1.0,0.95,36.935206,6.226995,0.630648,42.044118,13.425867,84.085470,2.772921,0.699259,0.061908,0.182321,0.062546
2,15,1.0,0.50,36.717006,5.854182,0.632830,42.514706,12.751738,84.051282,2.872405,0.698946,0.061705,0.181310,0.061957


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.30
  mean BER at threshold: 33.81%


,threshold,mean_ber_percent
0,0.30,33.810583
1,0.35,34.174145
2,0.25,34.290473
3,0.40,34.380970
4,0.15,34.588550
5,0.20,34.639077
6,0.10,35.496921
7,0.45,35.863876
8,0.50,37.039530
9,0.55,37.794118


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/mspc_lr.json


{'preprocess__sensor_mspc__pls__n_components': 20,
 'classifier__C': 0.1,
 'classifier__l1_ratio': 0.95}